# Step 1.2 — Extract the RAND HRS Codebook

**Purpose:** Extract searchable text from the RAND HRS Codebook PDF, preserving page numbers and recording extraction quality indicators.

Putting the text of the RAND HRS PDF codebook into a Delta table converts unstructured text into a structured, queryable database.

Because the RAND HRS dataset spans decades and contains thousands of longitudinal variables across multiple waves, trying to search or map these variables directly inside a raw PDF is incredibly difficult.

Loading this data into a Delta table serves several key purposes:

1. Powerful, Instant Searchability (SQL & DataFrames)

    Instead of manually scrolling through a 1,000-page PDF to find where a variable is defined, you can write simple SQL or PySpark queries.
    - Find a specific variable: You can instantly search thousands of pages for a variable like R10MSTAT.
    - Track variables across waves: You can filter the variable_like_tokens or text to see exactly which pages mention a specific topic across different years of the study.

2. Automated Pipeline Orchestration (Triage and OCR)

    As seen in your code, some pages will inevitably fail standard extraction (e.g., pages with charts, diagrams, or corrupted text encodings).
    - With a Delta table, you can easily isolate the failures. You can run a query to pull only the rows where review_flag == 'REVIEW_LOW_TEXT'.
    - You can then route only those specific pages to a heavy-duty OCR engine (like Azure Computer Vision or Tesseract), rather than wasting time and compute power processing the entire 1,000+ page PDF through OCR.

3. ACIDs Transactions, Time Travel, and Version Control

    Delta Lake provides features that a standard CSV or text file cannot:
    - Time Travel: If the RAND institute releases an updated version of the codebook and you overwrite or update your table, Delta Lake keeps a history. You can query exactly what your codebook data looked like last month.
    - Schema Enforcement: It ensures that every page processed always fits the exact structure (columns, data types) expected by your data pipeline.

4. Preparation for AI and LLMs (Retrieval-Augmented Generation)

    If your end goal involves advanced data analytics or building an AI assistant to help researchers understand the RAND HRS dataset:
    - A Delta table acts as the perfect storage layer for clean text chunks.
    - You can easily pass the page_text from this table into an embedding model to build a Vector Search index, allowing users to ask questions like, "Which variables measure cognitive decline in Wave 10?" and get answers directly from the codebook text.


**Outputs (Delta tables):**
- `dev_catalog.hrs_agent_metadata.hrs_document_pages` — one record per PDF page (chunking the document, one table record per page).

**Before running:** Update `PDF_PATH` in the configuration cell to the actual PDF path in a Unity Catalog Volume or another path accessible to this notebook.

This notebook creates a searchable extraction foundation. It does **not** claim that every variable definition or descriptive-statistics table has been perfectly structured. Later steps should parse and verify those records against the original PDF pages.

# Setup the Global variables

In [0]:
# Cell 1 — CONFIGURATION: Setup the common variables.

# Change PDF_PATH to the actual uploaded Codebook location.
# Recommended: a Unity Catalog Volume path such as:
# /Volumes/<catalog>/<schema>/<volume>/<filename>.pdf

PDF_PATH = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/randhrs1992_2022v1.pdf"

TARGET_CATALOG = "dev_catalog"
TARGET_SCHEMA = "hrs_agent_metadata"

PAGES_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.hrs_document_pages"
RUNS_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.hrs_document_processing_runs"

# Optional descriptive metadata. Update these values for the exact PDF edition.
DOCUMENT_ID = "rand_hrs_longitudinal_codebook"
DOCUMENT_TITLE = "RAND HRS Longitudinal Codebook"
CODEBOOK_VERSION = "1992-2022v1"  # e.g., edition/year, once verified

# If True, reprocess all pages and replace the existing page table contents.
# If False, this notebook still extracts the PDF but appends the run's pages.
REPLACE_PAGE_TABLE = True

print(f"PDF path: {PDF_PATH}")
print(f"Output pages table: {PAGES_TABLE}")
print(f"Output runs table: {RUNS_TABLE}")

# Confirm source file exists.

In [0]:
# Cell 2 — Confirm source file and install/import PDF extraction library
#
# PyMuPDF provides page-level text extraction. If the package is not available
# in the serverless environment, this cell attempts to install it for the session.

import os
import sys
import subprocess

# Setp 1: Confirm the PDF file (PDF_PATH) exists at the specified path. If not, raise an error.
if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH!r}. Update PDF_PATH to the actual file path. "
        "For Unity Catalog Volumes, use /Volumes/<catalog>/<schema>/<volume>/<file>.pdf."
    )

try:
    import pymupdf as fitz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymupdf"])
    import pymupdf as fitz

file_size_bytes = os.path.getsize(PDF_PATH)
print(f"Found PDF ({file_size_bytes:,} bytes). PyMuPDF version: {fitz.VersionBind}")

# Extract the Code Book page text into a array.

extracted_rows [ ]

In [0]:
# Cell 4 — Extract page text and assign review flags
# Store extracted pages in an array: extracted_rows[]

from datetime import datetime, timezone
import hashlib
import re
import uuid

run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc).replace(tzinfo=None)

with fitz.open(PDF_PATH) as pdf:
    source_page_count = pdf.page_count  # count the number of pages in the PDF file.
    extracted_rows = []                 # create an array object

    # For Loop.  Loop through each page and load each page into the "page" variable, then get the text and set the raw_text variable.
    for page_index in range(source_page_count):
        page = pdf.load_page(page_index)
        raw_text = page.get_text("text") or ""

        # Normalize line endings and remove trailing whitespace while retaining
        # meaningful line boundaries for later variable/table parsing.
        page_text = "\n".join(line.rstrip() for line in raw_text.replace("\r\n", "\n").replace("\r", "\n").split("\n")).strip()

        # Heuristic indicators only; these are not definitive classifications.
        # convert to upper text to make it Case Insensitivity.
        upper_text = page_text.upper()

        # Extracts every unique, uppercase word that looks like a RAND HRS variable name from the page, sorts them alphabetically, and saves them as a list.
        variable_tokens = sorted(set(re.findall(r"\b[A-Z][A-Z0-9]{1,11}\b", upper_text)))

        # checks if the current page contains any common statistical terms and sets a simple True or False flag.
        # It acts as a quick sensor to separate pages that contain actual data distribution summaries 
        # (like a table showing the mean or standard deviation of a variable) from pages that are just text descriptions or introduction notes.
        mentions_statistics = bool(re.search(
            r"\b(MEAN|MEDIAN|MINIMUM|MAXIMUM|STD\.? DEV|STANDARD DEVIATION|PERCENTILE|FREQUENCY|DISTRIBUTION)\b",
            upper_text
        ))

        # checks if the current page contains metadata headers or specific structural identifiers unique to the RAND HRS codebook, setting a simple True or False flag. [INDEX]
        # Its purpose is to instantly identify pages where a new variable definition or layout description begins, distinguishing them from general commentary pages or appendix text. [INDEX]
        mentions_variable = bool(re.search(
            r"\b(VARIABLE NAME|VARIABLE LABEL|VARIABLE DESCRIPTION|VARIABLE|R\d+[A-Z0-9_]*|HHIDPN)\b",
            upper_text
        ))

        # Mark pages that likely need OCR or visual review.
        text_length = len(page_text)
        if text_length < 40:
            review_flag = "REVIEW_LOW_TEXT"
        elif "\ufffd" in page_text:
            review_flag = "REVIEW_ENCODING"
        else:
            review_flag = "NONE"

        text_hash = hashlib.sha256(page_text.encode("utf-8")).hexdigest()

        # Takes all the information gathered for the current page and saves it as a structured record in your main storage list.
        # Because the code loops through the PDF page by page, extracted_rows.append() acts like filing a completed paperwork sheet into a master folder at the end of every loop cycle.
        extracted_rows.append({
            "run_id": run_id,
            "document_id": DOCUMENT_ID,
            "document_title": DOCUMENT_TITLE,
            "codebook_version": CODEBOOK_VERSION,
            "source_file_name": os.path.basename(PDF_PATH),
            "source_path": PDF_PATH,
            "page_number": page_index + 1,  # Human-readable, one-based page number
            "pdf_page_index": page_index,   # Zero-based PDF index for tooling
            "page_text": page_text,
            "character_count": text_length,
            "word_count": len(re.findall(r"\S+", page_text)),
            "text_sha256": text_hash,
            "mentions_variable_content": mentions_variable,
            "mentions_statistics_content": mentions_statistics,
            "variable_like_tokens": variable_tokens[:500],
            "review_flag": review_flag,
            "extracted_at": datetime.now(timezone.utc).replace(tzinfo=None)
        })

print(f"Extracted {len(extracted_rows):,} pages from {source_page_count:,} PDF pages.")
print("Pages flagged for review:", sum(1 for r in extracted_rows if r["review_flag"] != "NONE"))

# Store the extracted page text in Delta Table 
dev_catalog.hrs_agent_metadata.hrs_document_pages

In [0]:
# Cell 5 — Write extracted pages to Delta

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    BooleanType, ArrayType, TimestampType
)

# Schema defintion
# Used to define the blueprint or layout of a table
pages_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("codebook_version", StringType(), True),
    StructField("source_file_name", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("page_number", IntegerType(), False),
    StructField("pdf_page_index", IntegerType(), False),
    StructField("page_text", StringType(), True),
    StructField("character_count", IntegerType(), False),
    StructField("word_count", IntegerType(), False),
    StructField("text_sha256", StringType(), False),
    StructField("mentions_variable_content", BooleanType(), False),
    StructField("mentions_statistics_content", BooleanType(), False),
    StructField("variable_like_tokens", ArrayType(StringType()), True),
    StructField("review_flag", StringType(), False),
    StructField("extracted_at", TimestampType(), False)
])

# Create the Spark DataFrame and load the extracted page data.
# Takes your list of page records (extracted_rows) and transforms it into a distributed Spark DataFrame named "pages_df" based on the blueprint you defined in pages_schema.
pages_df = spark.createDataFrame(extracted_rows, schema=pages_schema)

# Write the data to the Delta Table.
# If true, then overwrite the table.  Do not append to it.
if REPLACE_PAGE_TABLE: 
    (pages_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(PAGES_TABLE))
else:
    (pages_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(PAGES_TABLE))

print(f"Wrote {pages_df.count():,} extracted page records to {PAGES_TABLE}")

# Record Processing Run

In [0]:
# Cell 6 — Record the processing run

from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, TimestampType

completed_at = datetime.now(timezone.utc).replace(tzinfo=None)
low_text_count = sum(1 for r in extracted_rows if r["review_flag"] != "NONE")

run_record = [{
    "run_id": run_id,
    "document_id": DOCUMENT_ID,
    "document_title": DOCUMENT_TITLE,
    "codebook_version": CODEBOOK_VERSION,
    "source_path": PDF_PATH,
    "source_file_size_bytes": int(file_size_bytes),
    "source_page_count": int(source_page_count),
    "extracted_page_count": int(len(extracted_rows)),
    "empty_or_low_text_page_count": int(low_text_count),
    "extraction_status": "SUCCESS" if len(extracted_rows) == source_page_count else "PARTIAL",
    "started_at": started_at,
    "completed_at": completed_at,
    "notes": "Page-level text extraction; review flags are heuristic and do not replace visual inspection."
}]

runs_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("codebook_version", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("source_file_size_bytes", LongType(), False),
    StructField("source_page_count", IntegerType(), False),
    StructField("extracted_page_count", IntegerType(), False),
    StructField("empty_or_low_text_page_count", IntegerType(), False),
    StructField("extraction_status", StringType(), False),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), False),
    StructField("notes", StringType(), True)
])

spark.createDataFrame(run_record, schema=runs_schema).write.format("delta").mode("append").saveAsTable(RUNS_TABLE)

print(f"Recorded processing run {run_id}")

# Report: page counts and pages needing review

In [0]:
# Cell 7 — Verify counts and identify pages needing review

from pyspark.sql import functions as F

display(
    spark.table(PAGES_TABLE)
         .filter(f"run_id = '{run_id}'")
         .select(
             "page_number", "character_count", "word_count",
             "mentions_variable_content", "mentions_statistics_content",
             "review_flag"
         )
         .orderBy("page_number")
         .limit(50)
)

print("Current run summary:")
display(
    spark.table(RUNS_TABLE)
         .filter(f"run_id = '{run_id}'")
)

print("Pages flagged for review:")
display(
    spark.table(PAGES_TABLE)
         .filter((F.col("run_id") == run_id) & (F.col("review_flag") != "NONE"))
         .select("page_number", "character_count", "review_flag", "page_text")
         .orderBy("page_number")
)

## Completion checklist

- [ ] `PDF_PATH` points to the actual Codebook PDF.
- [ ] The metadata schema can be created or already exists.
- [ ] The extracted page count matches the PDF page count.
- [ ] Pages flagged as low-text have been reviewed; scanned pages may need OCR.
- [ ] The extraction run is recorded in `hrs_document_processing_runs`.
- [ ] The original PDF remains unchanged.

## Limitations and next step

- `variable_like_tokens` is a heuristic token list, not a verified list of HRS variables.
- `mentions_variable_content` and `mentions_statistics_content` are search hints only.
- Page text extraction may not preserve complex table layouts accurately.
- This notebook does not yet create the normalized variable catalog, cohort coverage, wave coverage, or descriptive-statistics tables.

**Next notebook:** parse the extracted pages into candidate variable metadata and statistics, preserve page-level evidence, and validate the candidates against the PDF before marking them as authoritative.